In [1]:
from google import genai
from PIL import Image

%load_ext autoreload
%autoreload 2

In [2]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="global",  # changed from us-central1 previousy bc no access to gemini 3.5
)

### Put down the image file paths in batches

- Seperate into batches for ease of analysis (each batch kinda has their own charateristics so it's easy to improve my prompts if some batches perform poorly)

In [3]:
# 2. Local image file path, separated into different batches

path_batch1 = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/pink.png","test_images/combined.png",
         "test_images/neat.png", "test_images/neatest.png", "test_images/cropped-obvious.png"] # explore strength and weaknesses

path_batch2 = ["test_images/payslip1.jpeg", "test_images/payslip10.jpeg", "test_images/payslip2.png",
               "test_images/payslip20.png","test_images/payslip3.png", "test_images/payslip30.png"] # explore payslip with white space but different fonts

augmented_slips = ['test_images_augmented/by_technique/payslip2_rotation.png',
 'test_images_augmented/by_technique/payslip2_perspective.png',
 'test_images_augmented/by_technique/payslip2_elastic_distortion.png',
 'test_images_augmented/by_technique/payslip2_crop_and_pad.png',
 'test_images_augmented/by_technique/payslip2_jpeg_compression.png',
 'test_images_augmented/by_technique/payslip2_gaussian_blur.png',
 'test_images_augmented/by_technique/payslip2_shadow_vignette.png',
 'test_images_augmented/by_technique/payslip30_rotation.png',
 'test_images_augmented/by_technique/payslip30_perspective.png',
 'test_images_augmented/by_technique/payslip30_elastic_distortion.png',
 'test_images_augmented/by_technique/payslip30_crop_and_pad.png',
 'test_images_augmented/by_technique/payslip30_jpeg_compression.png',
 'test_images_augmented/by_technique/payslip30_gaussian_blur.png',
 'test_images_augmented/by_technique/payslip30_shadow_vignette.png']

augmented_rest = ['test_images_augmented/by_technique/payslip10_rotation.jpeg',
 'test_images_augmented/by_technique/payslip10_perspective.jpeg',
 'test_images_augmented/by_technique/payslip10_elastic_distortion.jpeg',
 'test_images_augmented/by_technique/payslip10_crop_and_pad.jpeg',
 'test_images_augmented/by_technique/payslip10_jpeg_compression.jpeg',
 'test_images_augmented/by_technique/payslip10_gaussian_blur.jpeg',
 'test_images_augmented/by_technique/payslip10_shadow_vignette.jpeg',
 'test_images_augmented/by_technique/payslip1_rotation.jpeg',
 'test_images_augmented/by_technique/payslip1_perspective.jpeg',
 'test_images_augmented/by_technique/payslip1_elastic_distortion.jpeg',
 'test_images_augmented/by_technique/payslip1_crop_and_pad.jpeg',
 'test_images_augmented/by_technique/payslip1_jpeg_compression.jpeg',
 'test_images_augmented/by_technique/payslip1_gaussian_blur.jpeg',
 'test_images_augmented/by_technique/payslip1_shadow_vignette.jpeg',
 'test_images_augmented/by_technique/payslip20_rotation.png',
 'test_images_augmented/by_technique/payslip20_perspective.png',
 'test_images_augmented/by_technique/payslip20_elastic_distortion.png',
 'test_images_augmented/by_technique/payslip20_crop_and_pad.png',
 'test_images_augmented/by_technique/payslip20_jpeg_compression.png',
 'test_images_augmented/by_technique/payslip20_gaussian_blur.png',
 'test_images_augmented/by_technique/payslip20_shadow_vignette.png',
 'test_images_augmented/by_technique/payslip3_rotation.png',
 'test_images_augmented/by_technique/payslip3_perspective.png',
 'test_images_augmented/by_technique/payslip3_elastic_distortion.png',
 'test_images_augmented/by_technique/payslip3_crop_and_pad.png',
 'test_images_augmented/by_technique/payslip3_jpeg_compression.png',
 'test_images_augmented/by_technique/payslip3_gaussian_blur.png',
 'test_images_augmented/by_technique/payslip3_shadow_vignette.png']


In [8]:
all_slips = augmented_rest + augmented_slips

## Ground truth labels

Set the true `label` ("tampered" / "authentic") and `label_signals` (true signal types present, if tampered) for each image in this batch. Stored in `image_labels.json`, keyed by image path.

`log_result()` looks these up by `image_path` automatically when logging — they are for analysis only and are **never** included in the prompt/model call. Run this once per image (re-running just overwrites with the same value).

- For the augmented images, the labels should've been created upon creation.

In [ ]:
# # Only need to set up after making a new orginal image once after it's been created.

# from result_logger import set_image_label, load_image_labels

# # Fill in the real ground truth for each image in `paths` below.
# # label: "tampered" or "authentic"
# # label_signals: list of signal types actually present (only meaningful if "tampered")
# #   -- see EXPECTED_SIGNAL_TYPES in result_logger.py for valid values

# ground_truth: dict[str, tuple[str, list[str]]] = {
#     paths[0]: ("authentic", []),
#     paths[1]: ("tampered", ["background_overlay", "font_weight_inconsistency",]),
#     paths[2]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[3]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[4]: ("tampered", ["font_weight_inconsistency","baseline_mismatch"]),
#     paths[5]: ("tampered", ["font_weight_inconsistency"]),
#     paths[6]: ("tampered", ["background_overlay", "font_weight_inconsistency", "baseline_mismatch"]),
# }

# for image_path, (label, label_signals) in ground_truth.items():
#     set_image_label(image_path, label, label_signals)

# load_image_labels()

In [4]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

# Read the V2 prompt (combined) and its system/task split, same pattern as V1

with open("prompt-library/V2.txt", "r", encoding="utf-8") as file:
    prompt_v2 = file.read()

with open("prompt-library/V2_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v2 = file.read()

with open("prompt-library/V2_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v2 = file.read()

<class 'str'>


## Result logging

Every model call below is logged to `notebook_results/results_log.jsonl` via `result_logger.log_result()`.
This keeps a permanent record across batches without cluttering this notebook.

**For each new batch of images:**
1. Bump `BATCH_ID` below (e.g. `"batch2"`)
2. Update `paths` in the image-loading cell to point at the new images
3. Re-run the experiment cells as usual — results from previous batches stay in the log

Use the "Review logged results" section at the end to compare across batches.

## Try with the same model for now to test the workflow
### Models (to double check)
- "gemini-2.5-flash"
- "gemini-2.5-pro"
- "gemini-3.5-flash"
- "gemini-3.1-pro-preview" because "gemini-3.5-pro" was not yet available on 12/06/2026 but releasing very soon

In [9]:
from exp_running import PromptVersion, run_experiment

# All prompt variants to sweep over. Add new versions here as new entries.
# prompt_versions = [
#     PromptVersion(prompt_id="V1", mode="combined", content=prompt_v1),
#     PromptVersion(prompt_id="V1_split", mode="split", system=system_prompt_v1, task=task_prompt_v1),
#     PromptVersion(prompt_id="V2", mode="combined", content=prompt_v2),
#     PromptVersion(prompt_id="V2_split", mode="split", system=system_prompt_v2, task=task_prompt_v2),
# ]

prompt_versions = [ PromptVersion(prompt_id="V2", mode="combined", content=prompt_v2),
    PromptVersion(prompt_id="V2_split", mode="split", system=system_prompt_v2, task=task_prompt_v2)
]

# Sweep config: edit these lists to control which combinations get run.
models = ["gemini-2.5-pro", "gemini-3.5-flash", "gemini-3.1-pro-preview"]  # Cut "gemini-2.5-flash"
temperatures = [0.1] #[0, 0.1, 0.2,0.8] -> 0.1 emerged as a victor

# Open images from the image paths and keep both for experimenting and logging
paths = all_slips
images = []

for path in paths:
    images.append(Image.open(path))

print(len(images))

image_list: list[tuple[str, Image.Image]] = list(zip(paths, images))

42


In [10]:
BATCH_ID = "augmented_slips"  # Verify this before running EVERYTIME

# Run the full sweep: every image x every prompt version x every model x every temperature.
for i in range(5):
    for model in models:
        for image_path, image in image_list:
            for prompt_version in prompt_versions:
                for temperature in temperatures:
                    run_experiment(client, image_path, image, prompt_version, model, temperature, BATCH_ID)

test_images_augmented/by_technique/payslip10_rotation.jpeg | V2 | gemini-2.5-pro | T=0.1 -> logged (21.67s)
test_images_augmented/by_technique/payslip10_rotation.jpeg | V2_split | gemini-2.5-pro | T=0.1 -> logged (22.12s)
test_images_augmented/by_technique/payslip10_perspective.jpeg | V2 | gemini-2.5-pro | T=0.1 -> logged (25.9s)
test_images_augmented/by_technique/payslip10_perspective.jpeg | V2_split | gemini-2.5-pro | T=0.1 -> logged (24.37s)
test_images_augmented/by_technique/payslip10_elastic_distortion.jpeg | V2 | gemini-2.5-pro | T=0.1 -> logged (20.8s)
test_images_augmented/by_technique/payslip10_elastic_distortion.jpeg | V2_split | gemini-2.5-pro | T=0.1 -> logged (24.05s)
test_images_augmented/by_technique/payslip10_crop_and_pad.jpeg | V2 | gemini-2.5-pro | T=0.1 -> logged (24.51s)
test_images_augmented/by_technique/payslip10_crop_and_pad.jpeg | V2_split | gemini-2.5-pro | T=0.1 -> logged (22.69s)
test_images_augmented/by_technique/payslip10_jpeg_compression.jpeg | V2 | gemini

KeyboardInterrupt: 

## Migrated the exp log review to analysis.ipynb for organisation
## Review logged results 

Load the full log (all batches) or just the current `BATCH_ID` to compare prompts/models without re-running anything.

In [26]:
import pandas as pd
from result_logger import load_results

df_all = load_results() # batch_id=BATCH_ID
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])
#df_all[["timestamp", "batch_id", "image_path", "prompt_id", "model", "verdict", "confidence", "signal_types", "format", "latency_s"]]

In [9]:
df_all['model'].value_counts()

model
gemini-2.5-pro            227
gemini-3.5-flash          148
gemini-3.1-pro-preview    148
gemini-2.5-flash           66
Name: count, dtype: int64

In [10]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format', 'augmentation'],
      dtype='object')

In [11]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589 entries, 0 to 588
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp        589 non-null    datetime64[ns, UTC]
 1   batch_id         589 non-null    object             
 2   image_path       589 non-null    object             
 3   prompt_id        589 non-null    object             
 4   model            589 non-null    object             
 5   temperature      589 non-null    float64            
 6   latency_s        589 non-null    float64            
 7   raw_response     589 non-null    object             
 8   parsed_response  588 non-null    object             
 9   notes            589 non-null    object             
 10  label            589 non-null    object             
 11  label_signals    589 non-null    object             
 12  verdict          588 non-null    object             
 13  confidence       588

In [12]:
# Calculate some accuracy, recall, F1 metrics
total_tests = len(df_all)
total_wrong_verdict = len(df_all[df_all['label'] != df_all['verdict']])

print("Wrong_verdict_percent", total_wrong_verdict/total_tests*100)

Wrong_verdict_percent 30.050933786078097


### Different batches will have different performances

In [13]:
df_all['verdict'].value_counts()

verdict
tampered        313
authentic       274
inconclusive      1
Name: count, dtype: int64

In [31]:
from analysis_util import compute_binary_metrics
total_matrics = compute_binary_metrics(df_all)
print(total_matrics)

precision           0.802778
recall              0.694712
specificity         0.721569
f1                  0.744845
accuracy            0.704918
true_positive     289.000000
false_positive     71.000000
false_negative    127.000000
true_negative     184.000000
excluded_count      2.000000
dtype: float64


#### Batch 1

In [15]:
batch1 = df_all[df_all['batch_id'] =='batch1']
print(compute_binary_metrics(batch1))

precision           0.964539
recall              0.814371
specificity         0.861111
f1                  0.883117
accuracy            0.822660
true_positive     136.000000
false_positive      5.000000
false_negative     31.000000
true_negative      31.000000
excluded_count      0.000000
dtype: float64


#### Batch 2

In [16]:
batch2 = df_all[df_all['batch_id'] =='batch2']
print(compute_binary_metrics(batch2))

precision           0.668605
recall              0.583756
specificity         0.695187
f1                  0.623306
accuracy            0.638021
true_positive     115.000000
false_positive     57.000000
false_negative     82.000000
true_negative     130.000000
excluded_count      2.000000
dtype: float64


#### Compare different prompt versions

In [17]:
df_all['prompt_id'].value_counts()

prompt_id
V1          178
V1_split    166
V2          124
V2_split    121
Name: count, dtype: int64

In [18]:
df_all.groupby('prompt_id').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_1643/994550254.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all.groupby('prompt_id').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
prompt_id,,,,,,,,,,
V1,0.820225,0.657658,0.753846,0.730000,0.693182,73.0,16.0,38.0,49.0,2.0
V1_split,0.802632,0.575472,0.750000,0.670330,0.638554,61.0,15.0,45.0,45.0,0.0
V2,0.794521,0.773333,0.693878,0.783784,0.741935,58.0,15.0,17.0,34.0,0.0
V2_split,0.786667,0.819444,0.673469,0.802721,0.760331,59.0,16.0,13.0,33.0,0.0


In [19]:
# Latency check: Latency is better?, To check again when have more results
print(df_all.groupby(['prompt_id'])['latency_s'].mean())
print(df_all.groupby(['prompt_id'])['latency_s'].std())

prompt_id
V1          16.808764
V1_split    15.575602
V2          20.997258
V2_split    20.882727
Name: latency_s, dtype: float64
prompt_id
V1           9.504234
V1_split     7.307306
V2          14.898995
V2_split    11.239944
Name: latency_s, dtype: float64


#### Model performances

In [20]:
df_all.groupby('model').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_1643/3643566522.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all.groupby('model').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
model,,,,,,,,,,
gemini-2.5-flash,0.750000,0.512195,0.708333,0.608696,0.584615,21.0,7.0,20.0,17.0,1.0
gemini-2.5-pro,0.781609,0.877419,0.464789,0.826748,0.747788,136.0,38.0,19.0,33.0,1.0
gemini-3.1-pro-preview,0.797297,0.702381,0.765625,0.746835,0.729730,59.0,15.0,25.0,49.0,0.0
gemini-3.5-flash,0.945946,0.416667,0.968750,0.578512,0.655405,35.0,2.0,49.0,62.0,0.0


#### Stability analysis
- Let's see how often do each model/ per batch/ per temperature change their predictions

In [49]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format'],
      dtype='object')

In [50]:
df_all['image_path'].value_counts()

image_path
test_images/payslip1.jpeg          62
test_images/payslip10.jpeg         62
test_images/payslip2.png           62
test_images/payslip20.png          62
test_images/payslip3.png           62
test_images/payslip30.png          62
test_images/First_Tamper.png       22
test_images/first-tamper.png       22
test_images/combined.png           20
test_images/neat.png               20
test_images/neatest.png            20
test_images/cropped-obvious.png    20
test_images/pink.png               18
Name: count, dtype: int64

In [51]:
verdict_proportions = df_all.groupby('image_path')['verdict'].value_counts(normalize=True)
verdict_proportions

image_path                       verdict     
test_images/First_Tamper.png     authentic       0.909091
                                 tampered        0.090909
test_images/combined.png         tampered        1.000000
test_images/cropped-obvious.png  tampered        0.900000
                                 authentic       0.100000
test_images/first-tamper.png     tampered        0.818182
                                 authentic       0.181818
test_images/neat.png             authentic       0.550000
                                 tampered        0.450000
test_images/neatest.png          authentic       0.600000
                                 tampered        0.400000
test_images/payslip1.jpeg        authentic       0.645161
                                 tampered        0.338710
                                 inconclusive    0.016129
test_images/payslip10.jpeg       authentic       0.770492
                                 tampered        0.229508
test_images/payslip2.png  

In [21]:
batch2.groupby('image_path').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_1643/2644542247.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batch2.groupby('image_path').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
image_path,,,,,,,,,,
test_images/First_Tamper.png,NaN,NaN,1.000000,NaN,1.000000,0.0,0.0,0.0,2.0,0.0
test_images/combined.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/cropped-obvious.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/first-tamper.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/neat.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/neatest.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/payslip1.jpeg,1.0,0.344262,NaN,0.512195,0.344262,21.0,0.0,40.0,0.0,1.0
test_images/payslip10.jpeg,0.0,NaN,0.770492,NaN,0.770492,0.0,14.0,0.0,47.0,1.0
test_images/payslip2.png,1.0,0.774194,NaN,0.872727,0.774194,48.0,0.0,14.0,0.0,0.0


### Check the augmented results
- Choose the slip images with strongest performance
- Augmented the slip images into the test_images_augmented/by_technique folder
- To augmented = load_results(batch = augmented_payslips) 
- Analyse to come up with V3 prompt/ find any weaknesses

In [27]:
augmented = load_results(batch_id = 'augmented_slips')

In [28]:
augmented

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format,augmentation
589,2026-06-15T09:44:35.446486+00:00,augmented_slips,test_images_augmented/by_technique/payslip2_ro...,V2_split,gemini-2.5-flash,0.1,13.69,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True,rotation
590,2026-06-15T09:44:48.969744+00:00,augmented_slips,test_images_augmented/by_technique/payslip2_pe...,V2_split,gemini-2.5-flash,0.1,13.52,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True,perspective
591,2026-06-15T09:45:00.624305+00:00,augmented_slips,test_images_augmented/by_technique/payslip2_el...,V2_split,gemini-2.5-flash,0.1,11.65,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True,elastic_distortion
592,2026-06-15T09:45:15.904342+00:00,augmented_slips,test_images_augmented/by_technique/payslip2_cr...,V2_split,gemini-2.5-flash,0.1,15.28,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True,crop_and_pad
593,2026-06-15T09:45:29.318397+00:00,augmented_slips,test_images_augmented/by_technique/payslip2_jp...,V2_split,gemini-2.5-flash,0.1,13.41,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True,jpeg_compression
594,2026-06-15T09:45:46.310427+00:00,augmented_slips,test_images_augmented/by_technique/payslip2_ga...,V2_split,gemini-2.5-flash,0.1,16.99,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",tampered,high,"[intra_string_baseline_shift, localized_backgr...",False,gaussian_blur
595,2026-06-15T09:46:03.212851+00:00,augmented_slips,test_images_augmented/by_technique/payslip2_sh...,V2_split,gemini-2.5-flash,0.1,16.90,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True,shadow_vignette
596,2026-06-15T09:46:20.554750+00:00,augmented_slips,test_images_augmented/by_technique/payslip30_r...,V2_split,gemini-2.5-flash,0.1,17.34,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True,rotation
597,2026-06-15T09:46:35.364757+00:00,augmented_slips,test_images_augmented/by_technique/payslip30_p...,V2_split,gemini-2.5-flash,0.1,14.81,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True,perspective
598,2026-06-15T09:46:56.666368+00:00,augmented_slips,test_images_augmented/by_technique/payslip30_e...,V2_split,gemini-2.5-flash,0.1,21.30,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,authentic,[],tampered,high,[categorical_font_mismatch],False,elastic_distortion


In [29]:
images = ["test_images/payslip2.png",
              'test_images_augmented/by_technique/payslip2_rotation.png',
 'test_images_augmented/by_technique/payslip2_perspective.png',
 'test_images_augmented/by_technique/payslip2_elastic_distortion.png',
 'test_images_augmented/by_technique/payslip2_crop_and_pad.png',
 'test_images_augmented/by_technique/payslip2_jpeg_compression.png',
 'test_images_augmented/by_technique/payslip2_gaussian_blur.png',
 'test_images_augmented/by_technique/payslip2_shadow_vignette.png',
 "test_images/payslip30.png", 
 'test_images_augmented/by_technique/payslip30_rotation.png',
 'test_images_augmented/by_technique/payslip30_perspective.png',
 'test_images_augmented/by_technique/payslip30_elastic_distortion.png',
 'test_images_augmented/by_technique/payslip30_crop_and_pad.png',
 'test_images_augmented/by_technique/payslip30_jpeg_compression.png',
 'test_images_augmented/by_technique/payslip30_gaussian_blur.png',
 'test_images_augmented/by_technique/payslip30_shadow_vignette.png']

 

In [32]:
augmented.groupby('image_path').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_15160/3519723449.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  augmented.groupby('image_path').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
image_path,,,,,,,,,,
test_images_augmented/by_technique/payslip2_crop_and_pad.png,1.0,0.75,NaN,0.857143,0.75,3.0,0.0,1.0,0.0,0.0
test_images_augmented/by_technique/payslip2_elastic_distortion.png,1.0,0.75,NaN,0.857143,0.75,3.0,0.0,1.0,0.0,0.0
test_images_augmented/by_technique/payslip2_gaussian_blur.png,1.0,0.75,NaN,0.857143,0.75,3.0,0.0,1.0,0.0,0.0
test_images_augmented/by_technique/payslip2_jpeg_compression.png,1.0,0.50,NaN,0.666667,0.50,2.0,0.0,2.0,0.0,0.0
test_images_augmented/by_technique/payslip2_perspective.png,1.0,0.75,NaN,0.857143,0.75,3.0,0.0,1.0,0.0,0.0
test_images_augmented/by_technique/payslip2_rotation.png,1.0,0.75,NaN,0.857143,0.75,3.0,0.0,1.0,0.0,0.0
test_images_augmented/by_technique/payslip2_shadow_vignette.png,1.0,0.75,NaN,0.857143,0.75,3.0,0.0,1.0,0.0,0.0
test_images_augmented/by_technique/payslip30_crop_and_pad.png,0.0,NaN,0.75,NaN,0.75,0.0,1.0,0.0,3.0,0.0
test_images_augmented/by_technique/payslip30_elastic_distortion.png,0.0,NaN,0.50,NaN,0.50,0.0,2.0,0.0,2.0,0.0
